In [ ]:
%pip install torch numpy scipy scikit-learn matplotlib

: 

### Plan
We should train/test before choosing the final 'best' model.
1. Import
2. Preprocessing: normalize data
3. Train/test with models. With parameter tuning with manual grid search. Find best number of past values to input.
4. Predict: 200 recursivily, scale back and output.
5. Get ready: report MAE and MSE, plot comparison.

### NN's all need time-series sequences: 
Long Short-Term Memory LSTM, a bit complex but sure good "LSTMs are designed to capture both short- and long-term dependencies in sequential data."
GRU is simpler and faster maybe worse
Use loss function just MSELoss probably
Optimizer idk. 

(python 3.13.7)

In [2]:
from scipy import io
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Load the MAT file
mat_data = io.loadmat('Xtrain.mat')

# Keys: dict_keys(['__header__', '__version__', '__globals__', 'Xtrain'])
# print(mat_data.keys())

data = mat_data['Xtrain'].flatten()
scaler = MinMaxScaler()                    # could chooose different scaler
data_scaled = scaler.fit_transform(data.reshape(-1,1)).flatten()

# Make time interval chunks for x and y
def create_intervals(data, interval_size):
    x, y = [],[]
    for i in range(len(data)-interval_size):
        x.append(data[i:i+interval_size])
        y.append(data[i+interval_size])
    return np.array(x), np.array(y)

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")       # torch cuda if possible (sure)

# LSTM
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        output, _ = self.lstm(x)
        output = output[:, -1, :]
        return self.fc(output)

# Training.. epochs can be increased ofc
def train_model(x,y,hidden_size,learning_rate,nepoch=20):
    x = torch.tensor(x, dtype=torch.float32).unsqueeze(-1).to(device)
    y = torch.tensor(y, dtype=torch.float32).unsqueeze(-1).to(device)

    model = LSTMModel(1, hidden_size).to(device)
    lossfunction = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(),lr=learning_rate)

    for epoch in range(nepoch):
        model.train()
        optimizer.zero_grad()
        output = model(x)
        loss = lossfunction(output,y)
        loss.backward()
        optimizer.step()
    
    return model, loss.item()

In [4]:
# Tuning on hidden_sizes, interval_size and learning_rate. Values? 
interval_sizes = [3,4,5,10,15,20,25,30,40]
hidden_sizes = [8,12,16,20,24,32,40,48]
learning_rates = [0.01,0.02,0.025,0.03,0.035,0.04,0.045,0.05]      # meer?

best_loss = float("inf")
best_config = None
best_model= None

# Try configurations
for interval in interval_sizes:
    x,y = create_intervals(data_scaled,interval)
    for hsize in hidden_sizes:
        for lr in learning_rates:
            model, loss = train_model(x,y,hsize,lr)
            print(f"interval={interval} hidden_size={hsize} learning_rate={lr} loss={loss}")
            # update best
            if (loss < best_loss):
                best_loss = loss
                best_config = (interval, hsize, lr)
                best_model= model

print(f"Best parameter configuration: {best_config}")
best_model.eval()

interval=3 hidden_size=8 learning_rate=0.01 loss=0.027397017925977707
interval=3 hidden_size=8 learning_rate=0.02 loss=0.03413770720362663
interval=3 hidden_size=8 learning_rate=0.025 loss=0.03919293358922005
interval=3 hidden_size=8 learning_rate=0.03 loss=0.03231048211455345
interval=3 hidden_size=8 learning_rate=0.035 loss=0.035083141177892685
interval=3 hidden_size=8 learning_rate=0.04 loss=0.03250301629304886
interval=3 hidden_size=8 learning_rate=0.045 loss=0.03772455081343651
interval=3 hidden_size=8 learning_rate=0.05 loss=0.03335808590054512
interval=3 hidden_size=12 learning_rate=0.01 loss=0.03172936290502548
interval=3 hidden_size=12 learning_rate=0.02 loss=0.03197171166539192
interval=3 hidden_size=12 learning_rate=0.025 loss=0.032526418566703796
interval=3 hidden_size=12 learning_rate=0.03 loss=0.033768460154533386
interval=3 hidden_size=12 learning_rate=0.035 loss=0.026151908561587334
interval=3 hidden_size=12 learning_rate=0.04 loss=0.02985823154449463
interval=3 hidden_

LSTMModel(
  (lstm): LSTM(1, 24, batch_first=True)
  (fc): Linear(in_features=24, out_features=1, bias=True)
)

First run: interval_sizes = [5,10,15,20,25,30], hidden_sizes = [32,40,48,56,64], learning_rates = [0.001,0.00075,0.0005] we got best parameter configuration: (5, 32, 0.001)
Second run: interval_sizes = [3,4,5,10,15,20,25,30,40] hidden_sizes = [16,24,32,40,48] learning_rates = [0.002,0.0015,0.001,0.0005] we got best parameter configuration: (20, 24, 0.002)     - conclusion lr can be bigger
Third run: interval_sizes = [3,4,5,10,15,20,25,30,40] hidden_sizes = [16,24,32,40,48] learning_rates = [0.003,0.0025,0.002,0.0015,0.001] we got best parameter configuration: (30, 48, 0.003)           - still change. 
Foruth: interval_sizes = [3,4,5,10,15,20,25,30,40] hidden_sizes = [16,24,32,40,48] learning_rates = [0.005,0.004,0.003,0.0025,0.002] we got best parameter configuration: (5, 48, 0.005)            - zo afhankelijk wat
Fifth: interval_sizes = [3,4,5,10,15,20,25,30,40] hidden_sizes = [16,24,32,40,48] learning_rates = [0.005,0.01] we got best parameter configuration: (5, 16, 0.01)            -lr moet gewoon veel hoger zijn
Sixth: interval_sizes = [3,4,5,10,15,20,25,30,40] hidden_sizes = [16,24,32,40,48] learning_rates = [0.005,0.01,0.05,0.1] best parameter configuration: (25, 16, 0.05)          - eindelijk lr range
Seventh: interval_sizes = [3,4,5,10,15,20,25,30,40] hidden_sizes = [8,12,16,20,24,32,40,48] learning_rates = [0.01,0.025,0.05,0.075,0.1] best parameter configuration: (25, 16, 0.025)      - chill top dit is het
Laatste: interval_sizes = [3,4,5,10,15,20,25,30,40] hidden_sizes = [8,12,16,20,24,32,40,48] learning_rates = [0.01,0.02,0.025,0.03,0.035,0.04,0.045,0.05] best parameter configuration: (20, 32, 0.04) - zo veranderlijk

In [6]:
# Prediction with 200 steps
input_sequence = data_scaled[-best_config[0]:]
preds = []

for _ in range(200):
    x = torch.tensor(input_sequence, dtype=torch.float32).unsqueeze(0).unsqueeze(-1).to(device)
    # reduce memory 
    with torch.no_grad():
        pred = model(x).item()
    preds.append(pred)
    # move to next
    input_sequence = np.append(input_sequence[1:],pred)

# scale back and output as csv
preds = scaler.inverse_transform(np.array(preds).reshape(-1,1)).flatten()
np.savetxt("200predictions.csv", preds, delimiter=",")

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

# May 8th test
test_data = io.loadmat("Xtest.mat")
yreal = test_data["Xtest"].flatten()

mae = mean_absolute_error(yreal[:200],preds)
mse = mean_squared_error(yreal[:200],preds)
print(f"MAE: {mae}")
print(f"MSE: {mse}")

# plot
plt.figure()
plt.plot(yreal[:200], label="Real")
plt.plot(preds, label="Predicted")
plt.legend()
plt.title("Actual and predicted laser measurements")
plt.show